In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

import sys
import os

sys.path.append(
    os.path.abspath("..")
)

from src.models import get_logistic_model, get_random_forest, get_xgboost
from src.evaluation import evaluate_model


In [5]:


df = pd.read_csv("../data/processed/fraud_clean.csv")

print(df.shape)
print(df["class"].value_counts())


(151112, 18)
class
0    136961
1     14151
Name: count, dtype: int64


In [ ]:
import pandas as pd

drop_cols = [
    "signup_time",
    "purchase_time",
    "device_id"
]

existing = [
    c
    for c in drop_cols
    if c in df.columns
]

df = df.drop(
    columns=existing
)

# Encode categorical columns
categorical = (
    df.select_dtypes(
        include=[
            "object",
            "string"
        ]
    ).columns
)

df = pd.get_dummies(
    df,
    columns=categorical,
    drop_first=True
)

# Replace inf
df = df.replace(
    [float("inf"), -float("inf")],
    pd.NA
)

# Fill missing values
for col in df.columns:

    if col == "class":
        continue

    if (
        str(df[col].dtype)
        in [
            "float64",
            "int64",
            "bool"
        ]
    ):
        df[col] = (
            df[col]
            .fillna(
                df[col].median()
            )
        )

df = df.fillna(0)

print(df.shape)
print(df.isna().sum().sum())



X = df.drop(
    columns=["class"]
)

y = df["class"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("\nBefore SMOTE")
print(
    y_train.value_counts()
)

smote = SMOTE(
    random_state=42
)

X_train, y_train = (
    smote.fit_resample(
        X_train,
        y_train
    )
)

print("\nAfter SMOTE")
print(
    y_train.value_counts()
)

models = {

    "Logistic Regression":
    get_logistic_model(),

    "Random Forest":
    get_random_forest(),

    "XGBoost":
    get_xgboost()

}

for name, model in models.items():

    model.fit(
        X_train,
        y_train
    )

    results = (
        evaluate_model(
            model,
            X_test,
            y_test
        )
    )

    print("\n================")

    print(name)

    print(
        results["report"]
    )

    print(
        "F1:",
        results["f1"]
    )

    print(
        "AUC-PR:",
        results["auc_pr"]
    )

(151112, 198)
0

Before SMOTE
class
0    109568
1     11321
Name: count, dtype: int64

After SMOTE
class
0    109568
1    109568
Name: count, dtype: int64

Logistic Regression
              precision    recall  f1-score   support

           0       0.95      0.68      0.79     27393
           1       0.18      0.68      0.28      2830

    accuracy                           0.68     30223
   macro avg       0.57      0.68      0.54     30223
weighted avg       0.88      0.68      0.74     30223

F1: 0.28328071077171596
AUC-PR: 0.29554196757627527

Random Forest
              precision    recall  f1-score   support

           0       0.95      0.99      0.97     27393
           1       0.83      0.53      0.64      2830

    accuracy                           0.95     30223
   macro avg       0.89      0.76      0.81     30223
weighted avg       0.94      0.95      0.94     30223

F1: 0.6445354602284975
AUC-PR: 0.6181134089186401

XGBoost
              precision    recall  f1-score 